In [0]:
#### PAYEMNTS
##### CREATED BY    : Pratik 564
##### CREATED DATE  : 20260817
##### DISCRIPTION   : Order payment transaction data containing payment method,installments, payment sequence, and transaction value.The Silver layer standardizes and validates payment information for financial and customer-order analytics.

##### MODIFIED BY   : Pratik **564**
##### MODIFIED DATE : 20260822

## Data Processing

#####- Standardize column names
#####- Trim string columns
#####-- Standardize payment type values
#####-- Cast payment sequence and installments to integer
#####-- Cast payment value to decimal/numeric
#####-- Validate payment values
#####-- Handle mandatory-field NULL values
#####-- Remove duplicate payment records
#####-- Add audit timestamp `ingest_ts`

READ CSV

In [0]:
%run "/Workspace/Users/pjadhav564@gmail.com/Brazil_project/functions"

In [0]:
path = "/Volumes/e_commerce_brazil/e_com_bronze/bronze_clean/"
payments_df = read_csv(f"{path}/order_payments")

In [0]:
#data cleaning and validation 


path = "/Volumes/e_commerce_brazil/e_com_bronze/bronze_clean/"
payments_df = read_csv(f"{path}/order_payments")


payments_df = clean_column_names(payments_df)
payments_df = trim_space_col(
    payments_df,
    [
        "order_id",
        "payment_type",
        "payment_installments",
        "payment_value",
    ],
)

payments_df = remove_null(
    payments_df,
    ["order_id"]
)

payments_df = handle_null(
    payments_df,
    {
        "payment_type": "unknown",
        "payment_sequential": 0,
        "payment_installments": 0,
        "payment_value": 0.0,
        "payment_type": "unknown"
    }
)
payments_df = drop_duplicates(payments_df, "order_id")
payments_df = payments_df = payments_df.withColumn(
    "payment_type", lower(col("payment_type"))
)
payments_df = cast_col(
    payments_df,
    {
        "payment_value": "double",
        "payment_installments": "integer",
        "payment_sequential": "integer",
    },
)
payments_df = sremove(
    payments_df,
    ["payment_sequential", "payment_installments", "payment_type"],
    pattern=r"\s+",
)

payments_df = update_ts(payments_df)

display(payments_df)

WRITE CSV


In [0]:
silver_path = "/Volumes/e_commerce_brazil/e_com_silver/updated_silver"

writefile(payments_df,(f'{silver_path}/payments_silver'),"overwrite")